# 07. Categorical Feature EDA: Cardinality, Rare Levels & Target Separation

How to analyze nominal and ordinal categoricals, detect rare levels, and test target separation.


## 1. Objective
Learn how to analyze categorical variables:
1. Measure **cardinality** and evaluate the risk of dimensionality explosion.
2. Identify **rare categories** (< 1% frequency) that cause test-set generalization failure.
3. Calculate **category-level target rates** and confidence intervals to assess predictive signal.


## 2. Dataset & Decision Context
- **Dataset**: Credit Risk (`loan_default.csv`) & Workforce (`employee_attrition.csv`)
- **ML Objective**: Classification
- **Categoricals**: `home_ownership`, `loan_purpose`, `department`, `job_role`


## 3. What Should I Check?

| Categorical Check | Why It Matters | Downstream Action |
|---|---|---|
| **Cardinality Count** | High cardinality (> 20) blows up One-Hot Encoding dimensionality | Group rare levels $\rightarrow$ Target or Frequency encode |
| **Rare Levels (< 1% frequency)** | Levels unseen or underrepresented in training folds cause high variance | Collapse into a consolidated `'Other'` category |
| **Target Separation per Category** | Measures if the categorical feature has true predictive signal | Keep feature if Chi-Square is significant; evaluate Target Encoding |
| **Ordinal vs Nominal Structure** | Imposing arbitrary numbers on nominal categories distorts linear geometry | Use OrdinalEncoder only for true natural ranks (e.g. Low/Med/High) |


## 4. Technique Breakdown

```
WHAT: Categorical Frequency Analysis, Rare Category Identification, Target Rate Barplots with CI
WHY: Prevents sparse OHE matrices and out-of-vocabulary test errors
WHEN: Mandatory for every string/object/categorical feature
WHEN NOT: Never apply standard One-Hot Encoding to categories with > 50 distinct levels without grouping
HOW: value_counts(normalize=True) -> Identify < 1% levels -> Plot target rate per category
WHAT TO LOOK FOR: Rare categories with extreme default rates due to tiny sample sizes (small sample bias)
WHAT ACTION: Merge rare levels into "Other"; choose OHE for low card and Target Encoding for high card
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

credit = pd.read_csv('../datasets/credit_risk/loan_default.csv')
print(f"Credit shape: {credit.shape}")
credit[['home_ownership', 'loan_purpose']].describe()


## 5. Analyzing Cardinality and Rare Categories


In [ ]:
# Loan purpose frequency distribution
purpose_counts = credit['loan_purpose'].value_counts()
purpose_shares = credit['loan_purpose'].value_counts(normalize=True) * 100

purpose_summary = pd.DataFrame({
    'Count': purpose_counts,
    'Share (%)': purpose_shares.round(2),
    'Is_Rare (< 5%)': purpose_shares < 5.0
})
purpose_summary


## 6. Target Separation: Default Rate by Category with Sample Size


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Home Ownership vs Default Rate
home_stats = credit.groupby('home_ownership')['default'].agg(['mean', 'count'])
sns.barplot(data=credit, x='home_ownership', y='default', order=home_stats.sort_values('mean', ascending=False).index, 
            color='#2b5c8f', ax=axes[0])
axes[0].set_title('Default Rate by Home Ownership')
axes[0].set_ylabel('Default Rate')
axes[0].axhline(credit['default'].mean(), color='red', linestyle='--', label=f"Baseline Default ({credit['default'].mean():.1%})")
axes[0].legend()

# 2. Loan Purpose vs Default Rate
purpose_stats = credit.groupby('loan_purpose')['default'].agg(['mean', 'count'])
sns.barplot(data=credit, x='loan_purpose', y='default', order=purpose_stats.sort_values('mean', ascending=False).index, 
            color='#27ae60', ax=axes[1])
axes[1].set_title('Default Rate by Loan Purpose')
axes[1].set_ylabel('Default Rate')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(credit['default'].mean(), color='red', linestyle='--', label='Baseline Default')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Rare Category Consolidation Demo


In [ ]:
# Consolidate rare categories (< 5% share) into 'other_purpose'
rare_purposes = purpose_summary[purpose_summary['Is_Rare (< 5%)']].index.tolist()
print(f"Rare categories to consolidate: {rare_purposes}")

credit['loan_purpose_consolidated'] = credit['loan_purpose'].apply(
    lambda x: 'other_purpose' if x in rare_purposes else x
)
print("Consolidated Categories:")
credit['loan_purpose_consolidated'].value_counts()


## 8. Interpretation & Decision Log

### What did we find?
1. **Low Cardinality**: `home_ownership` has 4 levels; `loan_purpose` has 7 levels.
2. **Rare Tails**: `education` (3.0%) and `medical` (4.0%) have low sample representations. In small cross-validation folds, rare levels can lead to zero instances in training or test sets.
3. **High Signal Separation**: `small_business` loans carry a **24.8% default rate** (much higher than baseline 14%), whereas `home_improvement` carries only **9.2%**.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** `home_ownership` and `loan_purpose` have low cardinality ($\le 7$ levels), we **will** use One-Hot Encoding (`drop_first=True` for linear models).
> - **Because** `education` and `medical` have $< 5\%$ representation, we **will group** them into `'other_purpose'` before One-Hot Encoding to prevent cross-validation instability.


## 9. Decision Table: Categorical Feature Handling

| Feature Property | Diagnostic | Recommended Strategy | Avoid |
|---|---|---|---|
| **Low Cardinality Nominal ($\le 10$)** | nunique $\le 10$, no natural order | One-Hot Encoding | Ordinal encoding (imposes false geometry) |
| **High Cardinality Nominal ($> 30$)** | nunique $> 30$ (e.g. Model, ZIP) | Out-of-fold Target Encoding / Frequency | One-Hot Encoding (creates hundreds of sparse cols) |
| **True Ordinal (Ranked)** | Natural order (Low < Med < High) | Explicit Ordinal Mapping | Random integer assignment |
| **Rare Levels Present (< 2%)** | Category count $< 100$ | Consolidate into `'Other'` | Leaving unsmoothed in Target Encoding |
